<img src="https://keystoneacademic-res.cloudinary.com/image/upload/c_pad,w_640,h_304/dpr_auto/f_auto/q_auto/v1/element/94/94774_thumb.png" width=300>

# Programação para Análise de Dados
## Aula 5 — Plano de Aula Consolidado

**Prof. Carlos E. Leal de Castro** e **Prof. Domingos Napolitano**| Duração: **1h 30min**

---

## 🎯 Objetivos da Aula
Ao final, os alunos serão capazes de:
- Identificar e classificar tipos de variáveis em um DataFrame
- Detectar e tratar valores faltantes (NaN) com estratégias adequadas
- Discretizar variáveis contínuas em categorias significativas

---

## ⏱️ Cronograma Geral

| Bloco | Tema | Tempo |
|-------|------|-------|
| 0 | Aquecimento e conexão com aulas anteriores | 5 min |
| 1 | **Tipos de Variáveis** (Aula 5) | 20 min |
| 2 | **Valores Faltantes** (Aula 6) | 30 min |
| 3 | **Discretização de Dados** (Aula 7) | 20 min |
| 4 | Exercícios Práticos | 10 min |
| 5 | Encerramento e próximos passos | 5 min |

---
## 🔷 Bloco 0 — Aquecimento (5 min)

**Pergunta para a turma:**
> "Quando vocês compram num e-commerce, quais informações sobre o pedido vocês imaginam que ficam salvas num banco de dados?"

Esperar respostas e mapear no quadro:
- Preço (R\$) → numérica contínua
- Quantidade → numérica discreta
- Categoria do produto → categórica
- Se foi entregue ou não → binária
- Data do pedido → data/hora

🔗 *"Hoje vamos aprender a lidar com exatamente esse tipo de dado — e o que fazer quando algum deles está faltando ou precisa ser simplificado."*

---
## 🔷 Bloco 1 — Tipos de Variáveis (20 min)

### 📌 Conceito Central

Em pandas, cada coluna de um DataFrame tem um **dtype** — o tipo de dado armazenado. Escolher o tipo correto melhora a performance e evita erros.

| Tipo | Exemplos | dtype pandas |
|------|----------|--------------|
| Numérica discreta | Nº de itens, cliques | `int64` |
| Numérica contínua | Preço, temperatura | `float64` |
| Categórica | Cidade, categoria | `category` |
| Binária | Ativo/Inativo, Sim/Não | `bool` |
| Data/Hora | Data de venda, nascimento | `datetime64` |

### 💻 Exemplo Prático — DataFrame com múltiplos tipos

In [1]:
import pandas as pd
import numpy as np

# DataFrame com múltiplos tipos de variáveis
df = pd.DataFrame({
    "produto":    ["A", "B", "C"],
    "categoria":  pd.Categorical(["bronze", "prata", "ouro"], ordered=True),
    "qtd":        [10, 5, 8],                        # discreta
    "preco":      [12.5, 20.0, 35.0],                # contínua
    "ativo":      [True, True, False],               # binária
    "data_venda": pd.to_datetime(["2025-08-10",
                                  "2025-08-11",
                                  "2025-08-12"]),    # data
})

display(df)
display(df.dtypes)   # verificar os tipos

,produto,categoria,qtd,preco,ativo,data_venda
0,A,bronze,10,12.5,True,2025-08-10
1,B,prata,5,20.0,True,2025-08-11
2,C,ouro,8,35.0,False,2025-08-12


produto               object
categoria           category
qtd                    int64
preco                float64
ativo                   bool
data_venda    datetime64[ns]
dtype: object

**Destaque:** `pd.Categorical(..., ordered=True)` indica que `ouro > prata > bronze`.  
Isso importa para ordenações e modelos de Machine Learning.

### 🔑 Conversão de Tipos — casos mais comuns

In [2]:
# --- Texto → Booleano ---
serie = pd.Series(["Sim", "Não", "Sim", "Não"])
booleano = serie.map({"Sim": True, "Não": False})

# --- Booleano → Inteiro (útil para modelos de ML) ---
inteiro = booleano.astype("int8")

print("Tipo booleano:", booleano.dtype)
print(booleano.values)

print("\nTipo inteiro:", inteiro.dtype)
print(inteiro.values)

Tipo booleano: bool
[ True False  True False]

Tipo inteiro: int8
[1 0 1 0]


In [3]:
# --- Strings → Datas ---
# Use errors='coerce' para não travar em datas inválidas (vira NaT)
datas_str = pd.Series(["2025-08-10", "2025-08-11", "invalido"])
datas = pd.to_datetime(datas_str, errors="coerce")
print(datas)

0   2025-08-10
1   2025-08-11
2          NaT
dtype: datetime64[ns]


### ⚠️ Atenção — Inteiros com NaN

> Nunca force `.astype('int')` numa coluna que contém `NaN` — Python vai gerar erro.  
> Use `Int64` (com I maiúsculo) — o tipo inteiro *nullable* do pandas:
>
> ```python
> df['coluna'].astype('Int64')
> ```

---
## 🔷 Bloco 2 — Valores Faltantes / NaN (30 min)

### 📌 Por que isso importa?

Dados faltantes são inevitáveis em projetos reais. Ignorá-los pode:
- Distorcer análises (viés)
- Quebrar modelos de Machine Learning
- Reduzir o poder estatístico dos resultados

---

### 🧩 Os 3 tipos de ausência

| Tipo | O que significa | Exemplo |
|------|----------------|---------|
| **MCAR** | Falta por acaso puro | Participante esqueceu de responder |
| **MAR** | Falta relacionada a outra variável observada | Homens não respondem perguntas emocionais |
| **MNAR** | Falta relacionada ao próprio valor ausente | Pessoas ricas não informam o salário |

*MCAR é o cenário ideal; MNAR é o mais problemático.*

---

### 💻 Identificando NaN

In [4]:
df_nan = pd.DataFrame({
    "preco":  [100, np.nan, 80, np.nan, 120],
    "cidade": ["SP", "RJ", np.nan, "BH", "SP"],
    "qtd":    [10, 5, np.nan, 8, 15],
})

# 1. Resumo geral — mostra contagem de não-nulos por coluna
print("--- .info() ---")
df_nan.info()

--- .info() ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preco   3 non-null      float64
 1   cidade  4 non-null      object 
 2   qtd     4 non-null      float64
dtypes: float64(2), object(1)
memory usage: 252.0+ bytes


In [5]:
# 2. Contagem de NaN por coluna
print("--- Contagem de NaN por coluna ---")
print(df_nan.isnull().sum())

# 3. Total de NaN no DataFrame inteiro
print("\n--- Total de NaN ---")
print(df_nan.isnull().sum().sum())

# 4. Percentual de ausentes
print("\n--- % de ausentes por coluna ---")
print((df_nan.isnull().sum() / len(df_nan) * 100).round(1))

--- Contagem de NaN por coluna ---
preco     2
cidade    1
qtd       1
dtype: int64

--- Total de NaN ---
4

--- % de ausentes por coluna ---
preco     40.0
cidade    20.0
qtd       20.0
dtype: float64


### 💻 Estratégias de Preenchimento

**Regra geral de escolha:**

| Variável | Estratégia recomendada |
|----------|----------------------|
| Numérica sem outliers | Média |
| Numérica com outliers | Mediana |
| Categórica | Moda |
| Série temporal ordenada | Interpolação |

In [6]:
df_trat = df_nan.copy()  # NUNCA altere o DataFrame original diretamente

# --- Média — para variáveis numéricas simétricas ---
df_trat["preco"] = df_trat["preco"].fillna(df_trat["preco"].mean())

# --- Mediana — mais robusta a outliers ---
df_trat["qtd"] = df_trat["qtd"].fillna(df_trat["qtd"].median())

# --- Moda — para variáveis categóricas ---
# .mode() retorna uma Series; [0] pega o valor mais frequente
moda_cidade = df_trat["cidade"].mode()[0]
df_trat["cidade"] = df_trat["cidade"].fillna(moda_cidade)

display(df_trat)
print("\nNaN restantes:", df_trat.isnull().sum().sum())

,preco,cidade,qtd
0,100.0,SP,10.0
1,100.0,RJ,5.0
2,80.0,SP,9.0
3,100.0,BH,8.0
4,120.0,SP,15.0



NaN restantes: 0


### 💻 Interpolação — ideal para séries temporais

In [7]:
df_temp = pd.DataFrame({
    "dia":         range(1, 8),
    "temperatura": [22, 24, np.nan, np.nan, 28, np.nan, 30]
})

# A interpolação linear "traça uma reta" entre os vizinhos para estimar o ausente
df_temp["temp_interp"] = df_temp["temperatura"].interpolate(method="linear")
print(df_temp)

   dia  temperatura  temp_interp
0    1         22.0    22.000000
1    2         24.0    24.000000
2    3          NaN    25.333333
3    4          NaN    26.666667
4    5         28.0    28.000000
5    6          NaN    29.000000
6    7         30.0    30.000000


### ⚠️ Boas práticas com NaN

1. **Nunca altere o DataFrame original** — sempre use `.copy()`
2. **Valide após o preenchimento:** `df.isnull().sum()` deve retornar `0`
3. **Documente** a estratégia escolhida — ela é uma decisão de negócio, não apenas técnica

---
## 🔷 Bloco 3 — Discretização de Dados (20 min)

### 📌 O que é e por que usar?

**Discretização** = transformar variável contínua → categorias (bins / caixas).

**Quando usar:**
- Para facilitar a comunicação com stakeholders ("clientes na faixa de 18–30 anos")
- Quando o modelo exige variáveis categóricas (ex: Naive Bayes)
- Para reduzir o impacto de outliers
- Para criar histogramas e análises segmentadas

---

### 💻 Preparando os dados

In [8]:
np.random.seed(42)
df_vendas = pd.DataFrame({
    "idade_cliente": np.random.randint(18, 70, size=20),
    "valor_compra":  np.random.uniform(20, 500, size=20)
})
print(df_vendas.head())

   idade_cliente  valor_compra
0             56    419.572468
1             69    121.922773
2             46    107.275984
3             32    108.034165
4             60    166.036277


### 💻 Equal Width — `pd.cut()`

Divide o intervalo em N partes de **mesma largura**.

$$Largura = \dfrac{valor_{máximo} - valor_{mínimo}}{N_{bins}}$$

**Desvantagem:** sensível a outliers — um valor extremo pode "esticar" um intervalo e deixar outros vazios.

In [9]:
# pd.cut() calcula automaticamente a largura dos intervalos
df_vendas["faixa_etaria_ew"] = pd.cut(
    df_vendas["idade_cliente"],
    bins=5,
    labels=["Jovem", "Jovem Adulto", "Adulto", "Adulto Maduro", "Sênior"]
)

print(df_vendas[["idade_cliente", "faixa_etaria_ew"]].head(10))
print("\nContagem por faixa (Equal Width):")
print(df_vendas["faixa_etaria_ew"].value_counts())

   idade_cliente faixa_etaria_ew
0             56   Adulto Maduro
1             69          Sênior
2             46          Adulto
3             32    Jovem Adulto
4             60          Sênior
5             25           Jovem
6             38    Jovem Adulto
7             56   Adulto Maduro
8             36    Jovem Adulto
9             40          Adulto

Contagem por faixa (Equal Width):
faixa_etaria_ew
Jovem            5
Adulto           5
Jovem Adulto     4
Adulto Maduro    4
Sênior           2
Name: count, dtype: int64


### 💻 Equal Frequency — `pd.qcut()`

Divide os dados em N grupos com **quantidade igual de observações** (usa quantis).

**Vantagem:** mais equilibrado entre grupos; não é afetado por outliers.

In [10]:
# pd.qcut() usa os quantis para garantir frequência igual em cada grupo
df_vendas["faixa_etaria_ef"] = pd.qcut(
    df_vendas["idade_cliente"],
    q=5,
    labels=["Jovem", "Jovem Adulto", "Adulto", "Adulto Maduro", "Sênior"]
)

print("Contagem por faixa (Equal Frequency):")
print(df_vendas["faixa_etaria_ef"].value_counts())

Contagem por faixa (Equal Frequency):
faixa_etaria_ef
Jovem            5
Adulto           5
Adulto Maduro    4
Jovem Adulto     3
Sênior           3
Name: count, dtype: int64


### Comparativo `pd.cut()` vs `pd.qcut()`

| | `pd.cut()` | `pd.qcut()` |
|--|-----------|------------|
| **Largura dos intervalos** | Igual | Variável |
| **Frequência por grupo** | Variável | Igual |
| **Sensível a outliers** | Sim | Não |
| **Quando usar** | Intervalos com significado claro | Distribuição equilibrada entre grupos |

---

### 💻 Quantis personalizados

In [11]:
# Separar os 10% menores, o "meio" e os 10% maiores
df_vendas["valor_segmento"] = pd.qcut(
    df_vendas["valor_compra"],
    q=[0, 0.1, 0.5, 0.9, 1.0],
    labels=["Muito Baixo", "Baixo", "Médio", "Alto"]
)

print("Distribuição por segmento de valor:")
print(df_vendas["valor_segmento"].value_counts())

Distribuição por segmento de valor:
valor_segmento
Baixo          8
Médio          8
Muito Baixo    2
Alto           2
Name: count, dtype: int64


---
## 🔷 Bloco 4 — Exercícios Práticos (10 min)

> Os exercícios cobrem os três temas de forma integrada, simulando um fluxo real de análise de dados.  
> Podem ser feitos individualmente ou em duplas.

---

### ✏️ Exercício 1 — Tipos de variáveis

Crie um DataFrame com pelo menos 5 colunas de tipos diferentes (numérica, categórica, binária, data). Em seguida:
1. Exiba o `.dtypes`
2. Converta a coluna categórica para `pd.Categorical(ordered=True)`
3. Converta a coluna binária para `int8`

In [12]:
# TODO — Exercício 1


---

### ✏️ Exercício 2 — Valores faltantes

Dado o DataFrame abaixo:
1. Quantos NaN existem por coluna?
2. Preencha `vendas` com a **mediana**, `regiao` com a **moda**, `preco` com **interpolação linear**
3. Confirme que não há mais NaN após o tratamento

In [13]:
# DataFrame base para o exercício
df_ex = pd.DataFrame({
    "vendas":   [500, np.nan, 300, np.nan, 450, 600],
    "regiao":   ["Norte", "Sul", np.nan, "Leste", "Sul", np.nan],
    "data":     pd.date_range("2025-01-01", periods=6),
    "preco":    [10.0, 12.5, np.nan, np.nan, 8.0, 9.0],
})
display(df_ex)

,vendas,regiao,data,preco
0,500.0,Norte,2025-01-01,10.0
1,NaN,Sul,2025-01-02,12.5
2,300.0,NaN,2025-01-03,NaN
3,NaN,Leste,2025-01-04,NaN
4,450.0,Sul,2025-01-05,8.0
5,600.0,NaN,2025-01-06,9.0


In [14]:
# TODO — Exercício 2


---

### ✏️ Exercício 3 — Discretização

Usando o DataFrame do Exercício 2 (após tratar os NaN):
1. Crie a coluna `faixa_vendas` com 3 categorias usando `pd.cut()`: `"Baixas"`, `"Médias"`, `"Altas"`
2. Crie a coluna `faixa_preco_eq` com 3 grupos de mesma frequência usando `pd.qcut()`
3. Exiba a contagem de cada grupo com `.value_counts()`

In [15]:
# TODO — Exercício 3


---
## 🔷 Bloco 5 — Encerramento (5 min)

### 🗺️ Resumo da Aula

```
DataFrame
├── Tipos de Variáveis
│   ├── Numéricas  → int64 / float64
│   ├── Categóricas → category / pd.Categorical
│   ├── Binárias   → bool / int8
│   └── Datas      → datetime64
│
├── Valores Faltantes (NaN)
│   ├── Identificar  → .isnull().sum()
│   ├── Preencher    → mean / median / mode / interpolate
│   └── Validar      → .isnull().sum() == 0
│
└── Discretização
    ├── pd.cut()  → Equal Width (mesma largura)
    └── pd.qcut() → Equal Frequency (mesma frequência)
```

---

### 📚 Para aprofundar
- McKinney, W. — *Python para Análise de Dados*, Cap. 7
- Carvalho et al. — *Ciência de Dados: Fundamentos e Aplicações*, Cap. Pré-processamento